#  Conceptos IA

---
## 1. Tokens
Un LLM no lee palabras, lee tokens. Vamos a estimar cuántos tokens tiene un texto y comparar español vs inglés.

In [1]:
# Estimación simple: ~1 token cada 4 caracteres en inglés, ~1.5 en español
def estimar_tokens(texto, idioma='es'):
    factor = 1.5 if idioma == 'es' else 1.0
    chars = len(texto)
    estimado = int(chars / 4 * factor)
    return {'caracteres': chars, 'tokens_estimados': estimado}

texto_es = 'El modelo de lenguaje procesa el texto dividiéndolo en fragmentos llamados tokens.'
texto_en = 'The language model processes text by splitting it into fragments called tokens.'

print('Español:', estimar_tokens(texto_es, 'es'))
print('English:', estimar_tokens(texto_en, 'en'))
print()
print('El mismo contenido consume más tokens en español.')
print('Esto impacta en costos y en el tamaño del contexto disponible.')

Español: {'caracteres': 82, 'tokens_estimados': 30}
English: {'caracteres': 79, 'tokens_estimados': 19}

El mismo contenido consume más tokens en español.
Esto impacta en costos y en el tamaño del contexto disponible.


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

WX_API_KEY        = os.getenv("WX_API_KEY", "")
WX_PROJECT_ID     = os.getenv("WX_PROJECT_ID", "")
WX_URL            = os.getenv("WX_URL", "https://us-south.ml.cloud.ibm.com")

print("Credenciales cargadas")

Credenciales cargadas


In [3]:
# Ahora veamos el uso REAL de tokens desde la API de watsonx
from ibm_watsonx_ai import APIClient, Credentials
from ibm_watsonx_ai.foundation_models import ModelInference

client = APIClient(Credentials(url=WX_URL, api_key=WX_API_KEY))

model = ModelInference(
    model_id="meta-llama/llama-3-3-70b-instruct",
    api_client=client,
    project_id=WX_PROJECT_ID,
)

response = model.generate(prompt=texto_es, params={'max_new_tokens': 10})
print('Tokens de input:', response['results'][0]['input_token_count'])
print('Tokens de output:', response['results'][0]['generated_token_count'])

Tokens de input: 23
Tokens de output: 10


---
## 2. Temperatura
Misma pregunta, diferente temperatura. Observá cómo cambia el output.

In [4]:
PROMPT = 'Dame tres ideas creativas para el nombre de una app de delivery de comida.'

for temp in [0.0, 0.7, 1.4]:
    response = model.generate_text(
        prompt=PROMPT,
        params={'temperature': temp, 'max_new_tokens': 200}
    )
    print(f'\nTEMPERATURA {temp}')
    print(response)


TEMPERATURA 0.0
 

1.  **Delicioso**: Este nombre sugiere que la comida entregada a través de la aplicación será deliciosa y atractiva, lo que puede generar expectación y deseo en los usuarios.
2.  **ComidaExpress**: Este nombre combina la idea de comida con la noción de rapidez y eficiencia, lo que puede atraer a usuarios que buscan una opción de entrega rápida y confiable.
3.  **Bocado**: Este nombre es corto y fácil de recordar, y sugiere la idea de un bocado o una porción de comida, lo que puede ser atractivo para usuarios que buscan una opción de entrega de comida rápida y fácil. 

Estos nombres pueden ser una buena opción para una aplicación de delivery de comida, ya que son fáciles de recordar, fáciles de pronunciar y transmiten una idea clara de lo que la aplicación ofrece. 



TEMPERATURA 0.7
 

1. **Appetito**: esta palabra sugiere hambre y apetito, lo que puede atraer a los usuarios a buscar comida a través de la aplicación.
2. **Delicioso**: este nombre puede evocar sentim

### ¿Cuándo usar qué temperatura?
| Tarea | Temperatura recomendada |
|---|---|
| Extracción de datos estructurados | 0.0 |
| Clasificación / routing | 0.0 - 0.2 |
| Respuestas de agente en producción | 0.1 - 0.3 |
| Generación de texto / resúmenes | 0.5 - 0.8 |
| Brainstorming / creatividad | 0.8 - 1.2 |

---
## 3. Embeddings y búsqueda semántica
Un embedding convierte texto en un vector numérico. Textos con significado similar tienen vectores cercanos.

In [5]:
import numpy as np

def similitud_coseno(v1, v2):
    """Qué tan 'cerca' están dos vectores. Resultado entre -1 y 1."""
    v1, v2 = np.array(v1), np.array(v2)
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

# Simulamos embeddings para mostrar el concepto
# (En producción estos vienen del modelo de embedding de watsonx)
frases = {
    'El auto está roto':          np.array([0.82, 0.15, 0.41, 0.55]),
    'El vehículo tiene una falla': np.array([0.79, 0.18, 0.44, 0.52]),
    'Mañana llueve en Buenos Aires': np.array([0.11, 0.87, 0.23, 0.09]),
    'Se espera lluvia para el fin de semana': np.array([0.09, 0.91, 0.19, 0.11]),
}

pares = [
    ('El auto está roto', 'El vehículo tiene una falla'),
    ('El auto está roto', 'Mañana llueve en Buenos Aires'),
    ('Mañana llueve en Buenos Aires', 'Se espera lluvia para el fin de semana'),
]

print('Similitud semántica entre frases:')
print('-' * 55)
for a, b in pares:
    sim = similitud_coseno(frases[a], frases[b])
    barra = '█' * int(sim * 20)
    print(f'{sim:.2f} {barra}')
    print(f'  A: "{a}"')
    print(f'  B: "{b}"')
    print()

Similitud semántica entre frases:
-------------------------------------------------------
1.00 ███████████████████
  A: "El auto está roto"
  B: "El vehículo tiene una falla"

0.37 ███████
  A: "El auto está roto"
  B: "Mañana llueve en Buenos Aires"

1.00 ███████████████████
  A: "Mañana llueve en Buenos Aires"
  B: "Se espera lluvia para el fin de semana"



In [6]:

from ibm_watsonx_ai.foundation_models import Embeddings

emb_model = Embeddings(
    model_id='ibm/slate-125m-english-rtrvr-v2',
    api_client=client,
    project_id=WX_PROJECT_ID
)

textos = [
    'El auto está roto',
    'El vehículo tiene una falla',
    'Mañana llueve en Buenos Aires'
]

vectores = emb_model.embed_documents(texts=textos)
print('Dimensiones del embedding:', len(vectores[0]))

sim_01 = similitud_coseno(vectores[0], vectores[1])
sim_02 = similitud_coseno(vectores[0], vectores[2])
print(f'Auto/Vehículo: {sim_01:.3f}')  # esperado: alto
print(f'Auto/Lluvia:   {sim_02:.3f}')  # esperado: bajo

Dimensiones del embedding: 768
Auto/Vehículo: 0.725
Auto/Lluvia:   0.570


---
## 4. Prompt Engineering — Patrones prácticos
Los 4 patrones que van a usar en los próximos 3 días.

In [7]:
# PATRÓN 1: Output estructurado
# Cuando necesitás que el agente devuelva datos procesables, no texto libre.

system_structured = '''Sos un clasificador de incidentes de soporte técnico.
Respondé SIEMPRE con un JSON válido con este esquema exacto:
{"categoria": str, "prioridad": "baja|media|alta|critica", "resumen": str}
No incluyas texto fuera del JSON.'''

user_structured = 'La app de producción no levanta después del deploy de las 14hs.'

# Respuesta esperada:
respuesta_esperada = {
    'categoria': 'DEPLOY',
    'prioridad': 'critica',
    'resumen': 'La aplicación de producción no inicia tras el último deploy'
}
import json
print('System prompt:')
print(system_structured)
print('\nRespuesta esperada:')
print(json.dumps(respuesta_esperada, indent=2, ensure_ascii=False))

System prompt:
Sos un clasificador de incidentes de soporte técnico.
Respondé SIEMPRE con un JSON válido con este esquema exacto:
{"categoria": str, "prioridad": "baja|media|alta|critica", "resumen": str}
No incluyas texto fuera del JSON.

Respuesta esperada:
{
  "categoria": "DEPLOY",
  "prioridad": "critica",
  "resumen": "La aplicación de producción no inicia tras el último deploy"
}


In [8]:
# PATRÓN 2: Chain of Thought
# Forzás al modelo a razonar antes de responder. Mejora precisión en tareas complejas.

system_cot = '''Antes de responder, razoná paso a paso dentro de etiquetas <thinking>.
Luego dá tu respuesta final dentro de etiquetas <answer>.
No saltees el paso de razonamiento.'''

user_cot = 'Un cliente compró un producto el 1 de enero con garantía de 90 días. Hoy es 5 de abril. ¿La garantía cubre el reclamo?'

respuesta_esperada_cot = '''
<thinking>
Fecha de compra: 1 de enero.
Garantía: 90 días.
Fecha de vencimiento: 1 enero + 90 días = 1 de abril.
Fecha del reclamo: 5 de abril.
5 de abril > 1 de abril → la garantía está vencida.
</thinking>
<answer>
No, la garantía no cubre el reclamo. Venció el 1 de abril y el reclamo es del 5 de abril.
</answer>
'''

print('Sistema:', system_cot)
print('\nRespuesta esperada:', respuesta_esperada_cot)

Sistema: Antes de responder, razoná paso a paso dentro de etiquetas <thinking>.
Luego dá tu respuesta final dentro de etiquetas <answer>.
No saltees el paso de razonamiento.

Respuesta esperada: 
<thinking>
Fecha de compra: 1 de enero.
Garantía: 90 días.
Fecha de vencimiento: 1 enero + 90 días = 1 de abril.
Fecha del reclamo: 5 de abril.
5 de abril > 1 de abril → la garantía está vencida.
</thinking>
<answer>
No, la garantía no cubre el reclamo. Venció el 1 de abril y el reclamo es del 5 de abril.
</answer>



In [9]:
# PATRÓN 3: Few-shot
# Ejemplos de comportamiento esperado. Más efectivo que describirlo.

system_fewshot = '''Clasificá incidentes de soporte según los siguientes ejemplos:

Input: "La app no carga"
Output: {"categoria": "UI", "prioridad": "media"}

Input: "Perdimos datos de producción"
Output: {"categoria": "DATA", "prioridad": "critica"}

Input: "El login tarda 30 segundos"
Output: {"categoria": "PERFORMANCE", "prioridad": "alta"}

Respondé SOLO con el JSON, sin texto adicional.'''

# El modelo va a generalizar el patrón a nuevos inputs
user_fewshot = 'El login falla para usuarios con caracteres especiales en el mail'

print('System (few-shot):', system_fewshot)
print('\nUser:', user_fewshot)
print('\nRespuesta esperada: {"categoria": "AUTH", "prioridad": "alta"}')

System (few-shot): Clasificá incidentes de soporte según los siguientes ejemplos:

Input: "La app no carga"
Output: {"categoria": "UI", "prioridad": "media"}

Input: "Perdimos datos de producción"
Output: {"categoria": "DATA", "prioridad": "critica"}

Input: "El login tarda 30 segundos"
Output: {"categoria": "PERFORMANCE", "prioridad": "alta"}

Respondé SOLO con el JSON, sin texto adicional.

User: El login falla para usuarios con caracteres especiales en el mail

Respuesta esperada: {"categoria": "AUTH", "prioridad": "alta"}
